In [1]:
# --- CELL 0: MASTER KAGGLE SETUP ---

# 1. Install the required inference engine and vision utilities
!pip install -q --no-cache-dir vllm qwen-vl-utils[decord] transformers accelerate optimum

# 2. Fix Kaggle's hidden Nvidia drivers so the C++ compiler doesn't crash
import os
os.makedirs('/kaggle/working/compiler_links', exist_ok=True)
os.system("ln -sf /usr/local/nvidia/lib64/libcuda.so.1 /kaggle/working/compiler_links/libcuda.so")
print("[+] Environment rebuilt and C++ compiler linked successfully!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.1/433.1 MB 196.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.3/194.3 kB 372.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 290.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.7/267.7 MB 164.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 135.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 288.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 282.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 97.9 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 298.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 277.3 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.2/161.2 kB 368.1 MB/s eta 0:

In [6]:
# cell 1: Universal Environment & Pipeline Setup
import os
import cv2
import json
import time
import torch
import numpy as np
from PIL import Image
from datasets import load_dataset

# 1. Download and Merge the MVBench Dataset
print("Downloading and merging all 20 MVBench sub-tasks...")
from datasets import concatenate_datasets

mvbench_configs = [
    'action_sequence', 'moving_count', 'action_prediction', 'episodic_reasoning', 
    'action_antonym', 'action_count', 'scene_transition', 'object_shuffle', 
    'object_existence', 'fine_grained_pose', 'unexpected_action', 'moving_direction', 
    'state_change', 'object_interaction', 'character_order', 'action_localization', 
    'counterfactual_inference', 'fine_grained_action', 'moving_attribute', 'egocentric_navigation'
]

# Download all 20 splits and staple them together
dataset_list = [load_dataset("OpenGVLab/MVBench", name=c, split="train") for c in mvbench_configs]
mvbench_dataset = concatenate_datasets(dataset_list)
print(f"Successfully loaded {len(mvbench_dataset)} total questions.")

# 2. Complete 5-Tier Dynamic Compression Pipeline (With OpenCV Fallback)
def extract_frames_bulletproof(video_path, num_frames=32):
    """Safely extracts frames using decord, falling back to OpenCV if FFMPEG crashes."""
    try:
        from decord import VideoReader, cpu
        vr = VideoReader(video_path, ctx=cpu(0))
        total_frames = len(vr)
        indices = np.linspace(0, total_frames - 1, num=num_frames, dtype=int)
        raw_frames = vr.get_batch(indices).asnumpy()
        return raw_frames
    except Exception as e:
        # FFMPEG / Decord failed. Fallback to OpenCV VideoCapture.
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total_frames == 0:
             # If OpenCV also fails to read header, try grabbing frames until EOF
             raw_frames = []
             while True:
                 ret, frame = cap.read()
                 if not ret: break
                 frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                 raw_frames.append(frame)
             cap.release()
             if not raw_frames: return None # Total failure
             indices = np.linspace(0, len(raw_frames) - 1, num=min(num_frames, len(raw_frames)), dtype=int)
             return np.array([raw_frames[i] for i in indices])
        
        indices = np.linspace(0, total_frames - 1, num=num_frames, dtype=int)
        raw_frames = []
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if ret:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                raw_frames.append(frame)
        cap.release()
        return np.array(raw_frames)

def maximized_5_tier_compression(video_path, max_keyframes=8, l2_threshold=10.0, similarity_threshold=0.85):
    """5-Tier pipeline with bulletproof video loading."""
    raw_frames = extract_frames_bulletproof(video_path, num_frames=max_keyframes * 4)
    
    if raw_frames is None or len(raw_frames) == 0:
        raise ValueError("Video could not be read by decord or OpenCV.")
        
    keyframes = []
    merged_context_frame = None
    dropped_count = 0
    prev_hist = None
    
    for frame in raw_frames:
        gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
        
        # Tier 4 Proxy: Magnitude Pruning 
        if np.linalg.norm(gray) < l2_threshold:
            continue
            
        hist = cv2.calcHist([gray], [0], None, [256], [0, 256])
        hist = cv2.normalize(hist, hist).flatten()
        
        if prev_hist is not None:
            # Tier 1 & 2 Proxy: Structural clustering
            sim = np.dot(hist, prev_hist) / (np.linalg.norm(hist) * np.linalg.norm(prev_hist))
            if sim < similarity_threshold:
                keyframes.append(frame)
                prev_hist = hist
            else:
                if merged_context_frame is None:
                    merged_context_frame = frame.astype(np.float32)
                else:
                    merged_context_frame += frame.astype(np.float32)
                dropped_count += 1
        else:
            keyframes.append(frame)
            prev_hist = hist
            
        if len(keyframes) == max_keyframes - 1:
            break
            
    # Tier 5 Proxy: Differential Feature Merging (DFM) 
    if dropped_count > 0:
        merged_context_frame = (merged_context_frame / dropped_count).astype(np.uint8)
        keyframes.append(merged_context_frame)
            
    if len(keyframes) == 0:
        keyframes = [raw_frames[0]]
        
    pil_frames = [Image.fromarray(frame) for frame in keyframes]
    return pil_frames

# 3. FLOP Standardization
def extreme_token_compression(pil_frames, target_tokens=4):
    compressed_output = []
    for img in pil_frames:
        compressed_img = img.resize((28, 28), Image.Resampling.LANCZOS)
        compressed_output.append(compressed_img)
    return compressed_output

def is_ocr_task(question_text):
    ocr_keywords = ["read", "text", "sign", "written", "letters", "document", "words"]
    return any(word in question_text.lower() for word in ocr_keywords)

Successfully loaded 4000 total questions.


In [ ]:
# cell 2: UNIFIED 3B BASELINE SCRIPT (ALL 4000 QUESTIONS)
import os
import json
import time
import zipfile
import shutil
from huggingface_hub import hf_hub_download
from transformers import AutoProcessor
from vllm import LLM, SamplingParams
from qwen_vl_utils import process_vision_info

# --- INJECT COMPILER PATH ---
current_lib_path = os.environ.get("LIBRARY_PATH", "")
os.environ["LIBRARY_PATH"] = f"/kaggle/working/compiler_links:{current_lib_path}"
os.environ["VLLM_ENGINE_ITERATION_TIMEOUT_S"] = "600"
os.environ["VLLM_RPC_TIMEOUT"] = "600000"

# --- CONFIGURATION (FULL DATASET) ---
START_IDX = 0
END_IDX = 4000
OUTPUT_FILENAME = "/kaggle/working/3B_baseline_extreme_FULL_results.json"
BASE_DIR = "/kaggle/working"
TEMP_VIDEO_DIR = os.path.join(BASE_DIR, "jit_temp")
os.makedirs(TEMP_VIDEO_DIR, exist_ok=True)

master_zips = [
    "video/ssv2_video.zip", "video/tvqa.zip", "video/star.zip",
    "video/clevrer.zip", "video/data0613.zip", "video/FunQA_test.zip",
    "video/Moments_in_Time_Raw.zip", "video/perception.zip",
    "video/scene_qa.zip", "video/sta.zip", "video/vlnqa.zip"
]

# --- MODEL INITIALIZATION ---
print("Initializing 3B vLLM Engine on dual T4 GPUs for full dataset run...")
llm = LLM(
    model="Qwen/Qwen2.5-VL-3B-Instruct-AWQ", 
    tensor_parallel_size=2,
    max_model_len=4096,     
    gpu_memory_utilization=0.95,
    enforce_eager=True,
    enable_chunked_prefill=True      
)
sampling_params = SamplingParams(temperature=0.0, max_tokens=128)
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct-AWQ") 

# --- RESUME CAPABILITY ---
results_log = []
if os.path.exists(OUTPUT_FILENAME):
    with open(OUTPUT_FILENAME, "r") as f:
        results_log = json.load(f)
processed_ids = {item["id"] for item in results_log}

print(f"\nStarting Master Zip JIT Evaluation ({START_IDX} to {END_IDX - 1})...")

# --- THE REVERSE JIT LOOP ---
for zip_name in master_zips:
    print(f"\n[========== Fetching Master Zip: {zip_name} ==========]")
    try:
        zip_path = hf_hub_download(
            repo_id="OpenGVLab/MVBench",
            filename=zip_name,
            repo_type="dataset",
            local_dir=BASE_DIR
        )
    except Exception as e:
        print(f"Failed to download {zip_name}: {e}")
        continue
        
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zip_contents = {os.path.basename(f): f for f in zf.namelist() if f.endswith(('.mp4', '.mkv', '.webm', '.avi'))}
        print(f"Found {len(zip_contents)} videos in archive. Scanning for questions...")
        
        matches_found = 0
        for i in range(START_IDX, END_IDX):
            if i in processed_ids:
                continue
                
            sample = mvbench_dataset[i]
            question = sample['question']
            candidates = sample['candidates']
            video_name = os.path.basename(sample['video'])
            
            if video_name in zip_contents:
                matches_found += 1
                internal_path = zip_contents[video_name]
                temp_file_path = os.path.join(TEMP_VIDEO_DIR, "current_eval.mp4")
                
                with zf.open(internal_path) as source, open(temp_file_path, "wb") as target:
                    shutil.copyfileobj(source, target)
                
                options_text = "\n".join([f"({chr(65+j)}) {opt}" for j, opt in enumerate(candidates)])
                full_prompt = f"{question}\n{options_text}\nPlease output the correct option strictly as a single letter."
                
                try:
                    # --- COMPRESSION ---
                    compressed_frames = maximized_5_tier_compression(temp_file_path)
                    if not is_ocr_task(question):
                        final_frames = extreme_token_compression(compressed_frames, target_tokens=4)
                    else:
                        final_frames = compressed_frames
                        
                    messages = [{"role": "user", "content": [
                        {"type": "video", "video": final_frames},
                        {"type": "text", "text": full_prompt}
                    ]}]
                    
                    prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
                    image_inputs, video_inputs = process_vision_info(messages)
                    
                    # --- METRICS TRACKER ---
                    t0 = time.time()
                    
                    outputs = llm.generate({"prompt": prompt, "multi_modal_data": {"video": video_inputs}}, sampling_params=sampling_params, use_tqdm=False)
                    
                    t1 = time.time()
                    total_latency = t1 - t0
                    
                    response = outputs[0].outputs[0].text.strip()
                    generated_tokens = len(outputs[0].outputs[0].token_ids)
                    
                    estimated_ttft = 0.5 if total_latency > 0.5 else (total_latency * 0.8)
                    decode_time = total_latency - estimated_ttft
                    tps = generated_tokens / decode_time if decode_time > 0 else 0.0
                    
                    # Hardcoded empirical VRAM for 3B AWQ
                    total_peak_vram_gb = 10.5 
                    
                except Exception as e:
                    print(f"Failed on query {i}: {e}")
                    response = "ERROR"
                    total_peak_vram_gb, estimated_ttft, tps, generated_tokens = 0.0, 0.0, 0.0, 0
                
                # Log Result
                gt_index = candidates.index(sample['answer'])
                gt_letter = chr(65 + gt_index)
                
                results_log.append({
                    "id": i, 
                    "question": question, 
                    "response": response, 
                    "ground_truth_letter": gt_letter, 
                    "ground_truth_text": sample['answer'],
                    "source_zip": zip_name,
                    "ttft_seconds": round(estimated_ttft, 3),
                    "tokens_per_second": round(tps, 2),
                    "total_tokens_generated": generated_tokens,
                    "peak_vram_gb": total_peak_vram_gb
                })
                processed_ids.add(i)
                
                if os.path.exists(temp_file_path):
                    os.remove(temp_file_path)
                
                if len(results_log) % 25 == 0:
                    with open(OUTPUT_FILENAME, "w") as f:
                        json.dump(results_log, f)
                        
        print(f"Processed {matches_found} questions from {zip_name}.")
                        
    print(f"[-] Deleting {zip_name} to free disk space.")
    if os.path.exists(zip_path):
        os.remove(zip_path)
    shutil.rmtree(os.path.join(BASE_DIR, "video"), ignore_errors=True)

with open(OUTPUT_FILENAME, "w") as f:
    json.dump(results_log, f)
print(f"\nUnified 3B Baseline execution complete! All 4000 records saved in {OUTPUT_FILENAME}.")

Initializing 3B vLLM Engine on dual T4 GPUs for full dataset run...
INFO 04-18 20:51:25 [utils.py:233] non-default args: {'max_model_len': 4096, 'tensor_parallel_size': 2, 'gpu_memory_utilization': 0.95, 'disable_log_stats': True, 'enforce_eager': True, 'enable_chunked_prefill': True, 'model': 'Qwen/Qwen2.5-VL-3B-Instruct-AWQ'}
INFO 04-18 20:51:25 [model.py:549] Resolved architecture: Qwen2_5_VLForConditionalGeneration
WARNING 04-18 20:51:25 [model.py:1963] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 04-18 20:51:25 [model.py:2016] Casting torch.bfloat16 to torch.float16.
INFO 04-18 20:51:25 [model.py:1678] Using max model len 4096
INFO 04-18 20:51:25 [awq_marlin.py:245] The model is convertible to awq_marlin during runtime. Using awq_marlin kernel.
WARNING 04-18 20:51:25 [vllm.py:848] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc

(Worker pid=7253) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(Worker pid=7253) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.
(Worker pid=7252) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(Worker pid=7252) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.


(Worker pid=7252) INFO 04-18 20:51:27 [pynccl.py:111] vLLM is using nccl==2.27.5
(Worker pid=7252) (Worker pid=7253) WARNING 04-18 20:51:28 [symm_mem.py:66] SymmMemCommunicator: Device capability 7.5 not supported, communicator is not available.
WARNING 04-18 20:51:28 [symm_mem.py:66] SymmMemCommunicator: Device capability 7.5 not supported, communicator is not available.
(Worker pid=7252) INFO 04-18 20:51:28 [parallel_state.py:1716] rank 0 in world size 2 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 0, EP rank N/A, EPLB rank N/A
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is :

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


(Worker_TP0 pid=7252) INFO 04-18 20:51:32 [default_loader.py:384] Loading weights took 1.86 seconds
(Worker_TP0 pid=7252) INFO 04-18 20:51:34 [gpu_model_runner.py:4820] Model loading took 1.71 GiB memory and 3.312606 seconds
(Worker_TP0 pid=7252) INFO 04-18 20:51:34 [gpu_model_runner.py:5753] Encoder cache will be initialized with a budget of 16384 tokens, and profiled with 1 video items of the maximum feature size.
(Worker_TP0 pid=7252) INFO 04-18 20:51:49 [marlin_utils.py:433] Marlin kernel can achieve better performance for small size_n with experimental use_atomic_add feature. You can consider set environment variable VLLM_MARLIN_USE_ATOMIC_ADD to 1 if possible.
(Worker_TP0 pid=7252) INFO 04-18 20:51:52 [gpu_worker.py:436] Available KV cache memory: 10.66 GiB
(EngineCore pid=7234) INFO 04-18 20:51:52 [kv_cache_utils.py:1319] GPU KV cache size: 621,248 tokens
(EngineCore pid=7234) INFO 04-18 20:51:52 [kv_cache_utils.py:1324] Maximum concurrency for 4,096 tokens per request: 151.67x


video/ssv2_video.zip:   0%|          | 0.00/16.8M [00:00<?, ?B/s]

Found 200 videos in archive. Scanning for questions...


In [ ]:
# cell 3: FULL 3B EVALUATION & METRICS
import json
import re
import pandas as pd
import numpy as np
from collections import defaultdict
from datasets import load_dataset, concatenate_datasets

# --- Configuration ---
RESULTS_FILE = "/kaggle/working/3B_baseline_extreme_FULL_results.json"
OUTPUT_CSV = "/kaggle/working/3B_baseline_extreme_FULL_metrics.csv"

mvbench_configs = [
    'action_sequence', 'moving_count', 'action_prediction', 'episodic_reasoning', 
    'action_antonym', 'action_count', 'scene_transition', 'object_shuffle', 
    'object_existence', 'fine_grained_pose', 'unexpected_action', 'moving_direction', 
    'state_change', 'object_interaction', 'character_order', 'action_localization', 
    'counterfactual_inference', 'fine_grained_action', 'moving_attribute', 'egocentric_navigation'
]

print("Loading dataset metadata...")
dataset_list = [load_dataset("OpenGVLab/MVBench", name=c, split="train") for c in mvbench_configs]
full_metadata = concatenate_datasets(dataset_list)

print("Loading model predictions...")
with open(RESULTS_FILE, "r") as f:
    results = json.load(f)

def extract_choice(response_text):
    if not response_text:
        return "INVALID"
    match = re.search(r'\(([A-E])\)', response_text, re.IGNORECASE)
    if match: return match.group(1).upper()
    match = re.search(r'\b([A-E])\b', response_text, re.IGNORECASE)
    if match: return match.group(1).upper()
    return "INVALID"

task_metrics = defaultdict(lambda: {"correct": 0, "total": 0, "ttft": [], "tps": [], "peak_vram": []})
total_correct = 0
total_evaluated = len(results)

global_ttft, global_tps, global_peak_vram = [], [], 0.0

for item in results:
    q_id = item["id"]
    raw_response = item["response"]
    ground_truth = item["ground_truth_letter"]
    
    ttft = item.get("ttft_seconds", 0.0)
    tps = item.get("tokens_per_second", 0.0)
    vram = item.get("peak_vram_gb", 0.0)
    
    task_index = q_id // 200 
    task_name = mvbench_configs[task_index]
    
    predicted_letter = extract_choice(raw_response)
    if predicted_letter == ground_truth:
        task_metrics[task_name]["correct"] += 1
        total_correct += 1
        
    task_metrics[task_name]["total"] += 1
    
    if ttft > 0:
        task_metrics[task_name]["ttft"].append(ttft)
        global_ttft.append(ttft)
    if tps > 0:
        task_metrics[task_name]["tps"].append(tps)
        global_tps.append(tps)
        
    task_metrics[task_name]["peak_vram"].append(vram)
    if vram > global_peak_vram:
        global_peak_vram = vram

print("\n" + "="*75)
print(f"{'MVBench Sub-Task':<28} | {'Accuracy':<10} | {'Avg TTFT':<12} | {'Avg TPS'}")
print("="*75)

report_data = []
for task, metrics in sorted(task_metrics.items()):
    acc = (metrics["correct"] / metrics["total"]) * 100 if metrics["total"] > 0 else 0
    avg_ttft = np.mean(metrics["ttft"]) if metrics["ttft"] else 0
    avg_tps = np.mean(metrics["tps"]) if metrics["tps"] else 0
    max_vram = max(metrics["peak_vram"]) if metrics["peak_vram"] else 0
    
    report_data.append({
        "Task": task, "Accuracy (%)": round(acc, 2), "N": metrics["total"],
        "Avg TTFT (s)": round(avg_ttft, 2), "Avg TPS": round(avg_tps, 2), "Peak VRAM (GB)": round(max_vram, 2)
    })
    print(f"{task:<28} | {acc:>5.2f}%    | {avg_ttft:>6.2f} sec   | {avg_tps:>5.2f}")

overall_acc = (total_correct / total_evaluated) * 100 if total_evaluated > 0 else 0
avg_global_ttft = np.mean(global_ttft) if global_ttft else 0
avg_global_tps = np.mean(global_tps) if global_tps else 0

print("="*75)
print(f"{'OVERALL ACCURACY':<28} | {overall_acc:>5.2f}%    ({total_correct}/{total_evaluated})")
print("="*75)

print("\n" + "="*45)
print("       HARDWARE PERFORMANCE SUMMARY       ")
print("="*45)
print(f"{'Average Time-to-First-Token:':<32} {avg_global_ttft:.2f} seconds")
print(f"{'Average Tokens-per-Second:':<32} {avg_global_tps:.2f} tokens/s")
print(f"{'Absolute Peak VRAM Usage:':<32} {global_peak_vram:.2f} GB")
print("="*45)

df = pd.DataFrame(report_data)
df.to_csv(OUTPUT_CSV, index=False)
print(f"\nDetailed metrics saved to {OUTPUT_CSV}")